Archival codes

In [ ]:
# At stay level
stay_overlap = (
    sf_mimicdata
    .loc[
        (sf_mimicdata["icu_mortality"] == 1) &
        (sf_mimicdata["mortality_after_discharge"] == 1),
        ["stay_id", "outtime", "death_abs_time"]
    ]
    .dropna()
    .drop_duplicates(subset="stay_id")
)

# time difference in hours
stay_overlap["death_minus_outtime_hours"] = (
    stay_overlap["death_abs_time"] - stay_overlap["outtime"]
).dt.total_seconds() / 3600

# Summary
stay_overlap["death_minus_outtime_hours"].describe(
    percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)


In [ ]:
stay_overlap["death_minus_outtime_hours"].hist(bins=50)

In [ ]:
# Identify admission_ids where both icu_mortality and mortality_after_discharge are True. These are admissions which probably have death very close to discharge (<=12 hours)
# means that they were likely icu_mortality but were marked as mortality_after_discharge due to inconsistencies in recording?

# Identify stay_ids where BOTH outcomes are 1
conflicting_stays = sf_mimicdata.loc[
    (sf_mimicdata["icu_mortality"] == 1) &
    (sf_mimicdata["mortality_after_discharge"] == 1),
    "stay_id"
].unique()

# Set post-discharge mortality = 0 for those stay_ids
sf_mimicdata.loc[
    sf_mimicdata["stay_id"].isin(conflicting_stays),
    "mortality_after_discharge"
] = 0

In [11]:
import pandas as pd

In [12]:
mimicdata = pd.read_parquet('../Dataset/mimic-iv/from_pipeline/ltm_grided_clipped_A_D_Z_Y_SelfPipeline.parquet')

In [13]:
mimicdata

,stay_id,grid_end,vent_mode__hours_since_last__last_12h,temperature__mean__last_12h,heart_rate__mean__last_12h,arterial_blood_pressure_mean__mean__last_12h,fluid_out_urine__mean__last_12h,pco2_arterial__mean__last_12h,respiratory_rate_measured__mean__last_12h,o2_saturation__mean__last_12h,...,death_abs_time,mortality_after_discharge,icu_mortality,t,ref_time,mortality_after_discharge_3M,A,D,Z,Y
0,mimic4-30000153,0 days 12:00:00,6.983333,37.625000,109.076920,84.380959,82.222221,44.0,17.230770,97.923080,...,NaT,0,False,0,2174-09-30 00:09:00,0,0,0.0,NaN,NaN
1,mimic4-30000153,1 days 00:00:00,NaN,37.333332,104.416664,91.846153,55.000000,NaN,12.500000,95.250000,...,NaT,0,False,1,2174-09-30 12:09:00,0,1,NaN,0.0,0.0
2,mimic4-30000213,0 days 12:00:00,NaN,36.911114,78.384613,78.846153,324.166656,41.5,21.692308,98.846153,...,2166-03-16,0,False,0,2162-06-21 17:38:00,0,0,0.0,NaN,NaN
3,mimic4-30000213,1 days 00:00:00,NaN,37.166668,85.250000,79.416664,230.000000,NaN,16.250000,99.083336,...,2166-03-16,0,False,1,2162-06-22 05:38:00,0,1,NaN,0.0,0.0
4,mimic4-30000484,0 days 12:00:00,NaN,35.629631,89.166664,64.750000,114.000000,59.0,15.000000,99.750000,...,2136-02-21,0,False,0,2136-01-15 05:23:32,1,0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599183,mimic4-39999858,1 days 12:00:00,NaN,36.777779,66.785713,77.250000,NaN,NaN,23.357143,93.071426,...,NaT,0,False,2,2167-04-28 00:36:00,0,0,0.0,NaN,NaN
599184,mimic4-39999858,2 days 00:00:00,NaN,36.944447,64.833336,77.250000,NaN,NaN,21.500000,95.166664,...,NaT,0,False,3,2167-04-28 12:36:00,0,0,0.0,NaN,NaN
599185,mimic4-39999858,2 days 12:00:00,NaN,36.902779,64.250000,81.818184,NaN,NaN,23.916666,92.550003,...,NaT,0,False,4,2167-04-29 00:36:00,0,0,0.0,NaN,NaN
599186,mimic4-39999858,3 days 00:00:00,NaN,36.870373,68.166664,71.166664,NaN,NaN,24.500000,93.250000,...,NaT,0,False,5,2167-04-29 12:36:00,0,0,0.0,NaN,NaN


In [15]:
mimicdata.groupby(['A', 'D', 'Z', 'Y'], dropna=False).size().reset_index(name='count')

,A,D,Z,Y,count
0,0,0.0,NaN,0.0,13
1,0,0.0,NaN,NaN,513542
2,0,1.0,NaN,1.0,6316
3,1,NaN,0.0,0.0,68201
4,1,NaN,1.0,1.0,11116


In [17]:
last_rows = (
    mimicdata
    .sort_values(['stay_id', 't'])
    .groupby('stay_id')
    .tail(1)
)

In [18]:
last_rows[['A', 'D', 'Z', 'Y']].drop_duplicates().reset_index(drop=True)

,A,D,Z,Y
0,1,NaN,0.0,0.0
1,1,NaN,1.0,1.0
2,0,1.0,NaN,1.0
3,0,0.0,NaN,0.0


In [5]:
mimicdata.loc[mimicdata.A==1].stay_id.unique()

array(['mimic4-30000153', 'mimic4-30000213', 'mimic4-30000484', ...,
       'mimic4-39999562', 'mimic4-39999810', 'mimic4-39999858'],
      shape=(79317,), dtype=object)

In [10]:
mimicdata.loc[mimicdata.stay_id=="mimic4-39999858"][["A", "D", "Z", "Y"]]

,A,D,Z,Y
599181,0,NaN,NaN,NaN
599182,0,NaN,NaN,NaN
599183,0,NaN,NaN,NaN
599184,0,NaN,NaN,NaN
599185,0,NaN,NaN,NaN
599186,0,NaN,NaN,NaN
599187,1,NaN,0.0,0.0
